In [3]:
!pip install jupyter_bokeh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.4 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [9]:
"""
California Housing – Panel Server App (Panel + Plotly)
-----------------------------------------------------

This is a **Panel server app**. Save as `app.py` and run:

    panel serve app.py --autoreload --show

Optionally export a self‑contained HTML from within the app via the sidebar button,
or from CLI:

    python app.py --save

Install deps:
    pip install panel plotly scikit-learn pandas numpy
"""

import argparse
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

import panel as pn
import plotly.express as px

pn.extension('plotly', 'tabulator')
# Use global sizing mode instead of passing to FastListTemplate (for compatibility)
pn.config.sizing_mode = 'stretch_width'

# ----------------------------- Load Data ----------------------------- #

def load_df() -> pd.DataFrame:
    data = fetch_california_housing(as_frame=True)
    df = data.frame.copy()
    # Add a categorical column similar to the original dataset's ocean proximity
    rng = np.random.default_rng(1337)
    df['ocean_proximity'] = rng.choice(
        ['<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'NEAR BAY'], size=len(df)
    )
    # Add a synthetic date column for time-based views
    dates = pd.date_range('2015-01-01', periods=len(df), freq='D')
    df['date'] = rng.choice(dates, len(df))
    # Rename to friendlier titles like the classic dataset
    rename_map = {
        'MedInc': 'median_income',
        'HouseAge': 'housing_median_age',
        'AveRooms': 'avg_rooms',
        'AveBedrms': 'avg_bedrooms',
        'Population': 'population',
        'AveOccup': 'avg_occupancy',
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        'MedHouseVal': 'median_house_value',
    }
    df = df.rename(columns=rename_map)
    return df

DF = load_df()

num_cols = DF.select_dtypes(include='number').columns.tolist()
cat_cols = DF.select_dtypes(exclude='number').columns.tolist()

def top_k_categories(s: pd.Series, k=12):
    counts = s.value_counts(dropna=True)
    keep = counts.nlargest(k).index
    return s.where(s.isin(keep), other='Other')

# ----------------------------- Widgets ----------------------------- #

default_target = 'median_house_value' if 'median_house_value' in DF.columns else num_cols[0]

x_metric = pn.widgets.Select(name='X metric (comparison)', options=num_cols, value='median_income')
y_metric = pn.widgets.Select(name='Y metric (comparison)', options=num_cols, value='housing_median_age')
color_by = pn.widgets.Select(name='Color by', options=[None] + cat_cols, value='ocean_proximity')
facet_by = pn.widgets.Select(name='Facet by (multi-facet)', options=[None] + cat_cols, value='ocean_proximity')

target_metric = pn.widgets.Select(name='Target metric (overview/trend/distribution)', options=num_cols, value=default_target)

# Numeric filters (2 sample sliders)
num_filters = []
for col in ['median_income', 'housing_median_age']:
    if col in DF.columns:
        vmin, vmax = float(DF[col].min()), float(DF[col].max())
        num_filters.append((col, pn.widgets.RangeSlider(name=f'{col} range', start=vmin, end=vmax, value=(vmin, vmax))))

# Date filter
if 'date' in DF.columns:
    dmin, dmax = pd.to_datetime(DF['date']).min(), pd.to_datetime(DF['date']).max()
    date_filter = pn.widgets.DateRangeSlider(name='date range', start=dmin, end=dmax, value=(dmin, dmax))
else:
    date_filter = None

# ----------------------------- Filtering ----------------------------- #

def apply_filters(df: pd.DataFrame) -> pd.DataFrame:
    out = df
    for col, slider in num_filters:
        lo, hi = slider.value
        out = out[(out[col] >= lo) & (out[col] <= hi)]
    if date_filter is not None and date_filter.value:
        start, end = date_filter.value
        out = out[(out['date'] >= pd.to_datetime(start)) & (out['date'] <= pd.to_datetime(end))]
    return out

filtered_df = pn.bind(apply_filters, DF)

# ----------------------------- Views ----------------------------- #

def overview_view(df: pd.DataFrame):
    if df.empty:
        return pn.pane.Markdown('**No rows match current filters.**')
    t = target_metric.value
    mean_val, med_val = float(df[t].mean()), float(df[t].median())
    stats = pn.Row(
        pn.indicators.Number(name='Rows', value=len(df), format='{value:,}'),
        pn.indicators.Number(name=f'Mean {t}', value=mean_val, format='{value:,.2f}'),
        pn.indicators.Number(name=f'Median {t}', value=med_val, format='{value:,.2f}')
    )
    fig = px.histogram(df, x=t, nbins=50, title=f'Distribution of {t}')
    fig.update_layout(height=420, hovermode='x unified')
    return pn.Column(stats, pn.pane.Plotly(fig, config={'displaylogo': False}), sizing_mode='stretch_width')


def trend_view(df: pd.DataFrame):
    if 'date' in df.columns:
        g = df.set_index('date')[target_metric.value].resample('MS').mean().reset_index()
        fig = px.line(g, x='date', y=target_metric.value, markers=True,
                      title=f'{target_metric.value} Trend Over Time (monthly mean)')
    else:
        fig = px.histogram(df, x=target_metric.value, nbins=50, title=f'{target_metric.value} Distribution')
    fig.update_layout(height=420, hovermode='x unified')
    return pn.pane.Plotly(fig, config={'displaylogo': False})


def comparison_view(df: pd.DataFrame):
    x, y = x_metric.value, y_metric.value
    color = color_by.value if color_by.value in df.columns else None
    hover = [c for c in ['latitude','longitude','ocean_proximity'] if c in df.columns]
    fig = px.scatter(df, x=x, y=y, color=color, hover_data=hover,
                     title=f'{y} vs {x}' + (f' colored by {color}' if color else ''))
    fig.update_traces(mode='markers', marker={'size': 6, 'opacity': 0.7})
    fig.update_layout(height=420)
    return pn.pane.Plotly(fig, config={'displaylogo': False})


def distribution_view(df: pd.DataFrame):
    t = target_metric.value
    facet = facet_by.value if facet_by.value in df.columns else None
    plot_df = df.copy()
    if facet is not None and plot_df[facet].nunique() > 12:
        plot_df[facet] = top_k_categories(plot_df[facet], k=12)
    fig = px.histogram(plot_df, x=t, color=facet, marginal='box', nbins=50,
                       title=f'Distribution of {t}' + (f' by {facet}' if facet else ''))
    fig.update_layout(barmode='overlay', height=420)
    fig.update_traces(opacity=0.65)
    return pn.pane.Plotly(fig, config={'displaylogo': False})


def multifacet_view(df: pd.DataFrame):
    x, y = x_metric.value, y_metric.value
    facet = facet_by.value if facet_by.value in df.columns else None
    if facet is None:
        return pn.pane.Markdown('Pick a **Facet by** column to see small multiples.')
    plot_df = df.copy()
    if plot_df[facet].nunique() > 12:
        plot_df[facet] = top_k_categories(plot_df[facet], k=12)
    fig = px.scatter(plot_df, x=x, y=y, facet_col=facet, facet_col_wrap=4,
                     title=f'Small Multiples: {y} vs {x} by {facet}', opacity=0.7)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig.update_layout(height=420)
    return pn.pane.Plotly(fig, config={'displaylogo': False})


def table_view(df: pd.DataFrame):
    if df.empty:
        return pn.pane.Markdown('**No rows to display.**')
    return pn.widgets.Tabulator(df.reset_index(drop=True), pagination='remote', page_size=20,
                                layout='fit_data_fill', selectable=True, show_index=False)

# ----------------------------- Layout (Template) ----------------------------- #

controls = [
    pn.pane.Markdown('### Controls'),
    target_metric,
    x_metric,
    y_metric,
    color_by,
    facet_by,
    pn.pane.Markdown('### Filters'),
]

for _, slider in num_filters:
    controls.append(slider)
if date_filter is not None:
    controls.append(date_filter)

save_btn = pn.widgets.Button(name='Export self-contained HTML', button_type='primary')
save_status = pn.pane.Markdown('')

SAVE_NAME = 'california_housing_dashboard.html'


def _do_save(event=None):
    try:
        template = make_template()  # capture current state
        template.save(SAVE_NAME, embed=True)
        save_status.object = f'✅ Saved **{SAVE_NAME}** in current folder.'
    except Exception as e:
        save_status.object = f'⚠️ Save failed: {e}'

save_btn.on_click(_do_save)
controls.extend([pn.layout.Divider(), save_btn, save_status])


def make_template():
    return pn.template.FastListTemplate(
        title='California Housing – Panel + Plotly Dashboard',
        sidebar=controls,
        main=[
            pn.Tabs(
                ('Overview', pn.bind(overview_view, filtered_df)),
                ('Trend', pn.bind(trend_view, filtered_df)),
                ('Comparison', pn.bind(comparison_view, filtered_df)),
                ('Distribution', pn.bind(distribution_view, filtered_df)),
                ('Multi‑facet', pn.bind(multifacet_view, filtered_df)),
                ('Table', pn.bind(table_view, filtered_df)),
            ),
        ],
        sidebar_width=320,
        accent_base_color='#2563eb',
        header_background='#111827'
    )

app = make_template()
app.servable()

# ----------------------------- CLI Save Option ----------------------------- #

parser = argparse.ArgumentParser()
parser.add_argument('--save', action='store_true', help='Save a standalone HTML and exit')
args, _ = parser.parse_known_args()

if __name__ == '__main__':
    if args.save:
        app.save(SAVE_NAME, embed=True)
        print(f'Saved {SAVE_NAME}')


In [15]:
app

FastListTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [browser_info] BrowserInfo(dark_mode=True)
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [main-133237673300992] Tabs(sizing_mode='stretch_width')
        [0] ParamFunction(function, _pane=Column, defer_load=False, name='Overview', sizing_mode='stretch_width')
        [1] ParamFunction(function, _pane=Plotly, defer_load=False, name='Trend', sizing_mode='stretch_width')
        [2] ParamFunction(function, _pane=Plotly, defer_load=False, name='Comparison', sizing_mode='stretch_width')
        [3] ParamFunction(function, _pane=Plotly, defer_load=False, name='Distribution', sizing_mode='stretch_width')
        [4] ParamFunction(function, _pane=Plotly, defer_load=False, name='Multi‑facet', sizing_mode='stretch_width')
        [5] ParamFunction(function, _pane=Tabulator, defer_load=False, name='Table', sizing_mode='stretch_width')
    [nav-133237673297440] Markdown(str, sizing_mode='stretch_width')
    [nav-133237676090064] Select(name='Target metric (..., options=['median_income', ...], sizing_mode='stretch_width', value='median_house_value')
    [nav-133237683290624] Select(name='X metric (comparison)', options=['median_income', ...], sizing_mode='stretch_width', value='median_income')
    [nav-133237724013408] Select(name='Y metric (comparison)', options=['median_income', ...], sizing_mode='stretch_width', value='housing_median_age')
    [nav-133237678423840] Select(name='Color by', options=[None, 'ocean_proximity', ...], sizing_mode='stretch_width', value='ocean_proximity')
    [nav-133237676088336] Select(name='Facet by (multi-facet)', options=[None, 'ocean_proximity', ...], sizing_mode='stretch_width', value='ocean_proximity')
    [nav-133237724013360] Markdown(str, sizing_mode='stretch_width')
    [nav-133237676090736] RangeSlider(end=15.0001, name='median_income range', sizing_mode='stretch_width', start=0.4999, value=(0.4999, 15.0001), value_end=15.0001, value_start=0.4999)
    [nav-133237673690128] RangeSlider(end=52.0, name='housing_median_age r..., sizing_mode='stretch_width', start=1.0, value=(1.0, 52.0), value_end=52.0, value_start=1.0)
    [nav-133237673689744] DateRangeSlider(end=Timestamp('2071-07-05 0..., name='date range', sizing_mode='stretch_width', start=Timestamp('2015-01-01 0..., value=(Timestamp('2015-01-01 00:..., value_end=Timestamp('2071-07-05 0..., value_start=Timestamp('2015-01-01 0...)
    [nav-133237673298160] Divider(sizing_mode='stretch_width')
    [nav-133237676089344] Button(button_type='primary', name='Export self-contained H..., sizing_mode='stretch_width')
    [nav-133237673298064] Markdown(str, sizing_mode='stretch_width')

In [17]:
# ✅ Save a fully self-contained HTML snapshot (Option B)
import panel as pn

# Wrap the template’s main content in a plain layout
layout = pn.Column(*app.main)

# Save as a self-contained HTML with embedded state and inline JS/CSS
layout.save("california_housing_dashboard_embed.html", embed=True, resources="inline")

# Download it from Colab to your computer
from google.colab import files
files.download("california_housing_dashboard_embed.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>